------Crimeのデータについて------

手順 1, 各年のcrime2020～2024を縦結合

手順 2, 犯罪辞書と照合し，分類する

手順 3, 年別，コミュニティエリア別の集計表を作る


------ACSのデータについて--------

手順 1, Tractごとのデータからコミュニティx年別のデータの集計表を作成する

手順 2, 複数のデータセットを横結合する

※　Forループで行う

------最後に------------------

手順 1, 上記2つのデータセットをヘッダーに結合して完成

In [2]:
import pandas as pd
import numpy as np

# まずは，crime のデータに分類辞書を結合する

crime_dict = pd.read_csv(r"crime_dictionary.csv")

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
dfs = {}

for year in range(2020, 2024+1):
    dfs[year] = pd.read_csv(rf"Crimes{year}.csv") 
    # dfs.append(df)

crime = pd.concat(dfs.values(), ignore_index=True)

In [5]:
crime_merge = pd.merge(crime, crime_dict, on = "primary_type", how = "left")

In [6]:
# crime_merge.head(50)

In [7]:
crime_merge.rename(columns = {"crime_group1_prime" : "crimetype"}, inplace = True)

crime_summary = (
    crime_merge
    .groupby(["year", "community_area", "crimetype"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print(crime_summary)

print(crime_summary)

crimetype  year  community_area  other1  property1  public_order1  violent1
0          2020             1.0     180       1947            179      1002
1          2020             2.0     165       1919            161       866
2          2020             3.0     131       1814            169       851
3          2020             4.0      72       1313             78       431
4          2020             5.0      59        850             61       191
..          ...             ...     ...        ...            ...       ...
380        2024            73.0     260       1696            250       863
381        2024            74.0      57        309             27       156
382        2024            75.0     199       1036            158       619
383        2024            76.0      89       1014            196       397
384        2024            77.0     185       2292            180       917

[385 rows x 6 columns]
crimetype  year  community_area  other1  property1  public_order

In [8]:
header = pd.read_csv("header.csv")

In [9]:
header = header[header["year"] >= 2020]

In [10]:
header.to_csv("header.csv", index = False)

In [11]:
# # ACSのデータセットを読み込む
# dataset = "B17001"
# name = ""
df1 = pd.read_csv(r"ACSB01001_2020-2024.csv")
df1.head()

,year,malepop,male15_19pop,male15_24pop,male15_29pop,geoid,namelsad,tractpop,tractce,community_area,weight,comm_pop,pop_share,community,commarea_km2
0,2020,1793,181,213,287,17031510300,Census Tract 5103,4805,510300,51,1.0,15144,0.317287,SOUTH DEERING,303.797060
1,2020,1182,155,213,294,17031520100,Census Tract 5201,2219,520100,52,1.0,22999,0.096482,EAST SIDE,83.241728
2,2020,1663,200,321,484,17031590700,Census Tract 5907,3064,590700,59,1.0,15246,0.200971,MCKINLEY PARK,39.431800
3,2020,2008,109,349,854,17031600400,Census Tract 6004,3736,600400,60,1.0,32901,0.113553,BRIDGEPORT,58.291519
4,2020,2233,39,192,395,17031830600,Census Tract 8306,4923,830600,1,1.0,55643,0.088475,ROGERS PARK,51.259902


In [12]:
# 集計対象列（後ろ10列を除く）
sum_cols = df1.columns[:-10]

# year, community_area はグループキーとして残すので除外
sum_cols = [col for col in sum_cols if col not in ["year", "community_area"]]

sum_cols
# 集計
result = (
    df1.groupby(["year", "community_area"], as_index=False)[sum_cols]
      .sum()
)

result.head()

,year,community_area,malepop,male15_19pop,male15_24pop,male15_29pop
0,2020,1,27209,1249,3543,6025
1,2020,2,39092,2638,4753,8354
2,2020,3,30022,817,2711,6786
3,2020,4,19735,590,1305,3198
4,2020,5,17723,627,1184,2530


In [13]:
import subprocess
import re

# Git管理されている _2020-2024.csv ファイルを取得
files = subprocess.check_output(
    ["git", "ls-files", "*_2020-2024.csv"],
    text=True
).splitlines()

# ACS_ と _2020-2024.csv の間を抽出
acs_tables = [
    re.search(r"ACS(.*?)_2020-2024\.csv", f).group(1)
    for f in files
    if re.search(r"ACS(.*?)_2020-2024\.csv", f)
]

print(acs_tables)

['B01001', 'B02008', 'B02009']
